# Token Neighbourhood Analysis: GPT-2 Medium

**Date:** 2026-03-25  
**Depends on:** Stage 1 results (`output/stage1_results.pt`)  
**Outputs:** `results/` directory

## Purpose

Extract the BPE vocabulary indices for basin attractor and waypoint tokens discovered
in Stage 1, then examine their **embedding neighbourhoods** in `W_E` to determine whether
terminal tokens cluster semantically or by BPE substring adjacency.

### GPT-2 Medium Context

Stage 1 showed **total gravitational collapse** to a single basin token `D` (100% of 125 prompts).
Waypoint tokens observed during dissolution: `local`, `national`, `RAW`, `BR`, `AB`.

---

In [ ]:
# ============================================================
# STEP 0: OUTPUT DIRECTORY SETUP
# ============================================================
import os
from pathlib import Path

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)
(OUTPUT_DIR / "data").mkdir(exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# ============================================================
# STEP 1: SETUP — Load the model
# ============================================================
import torch
import numpy as np
import json
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-medium", device=device)
print(f"Running on: {device}")
print(f"Vocabulary size: {model.cfg.d_vocab}")
print(f"Embedding dimension: {model.cfg.d_model}")

---
## 2. Token ID Extraction

Basin token: `D` (the sole attractor).  
Waypoint tokens: `local`, `national`, `RAW`, `BR`, `AB` (observed during dissolution).

In [ ]:
# ============================================================
# STEP 2: TOKEN ID EXTRACTION
# ============================================================

# Basin attractor tokens (terminal states from Stage 1)
BASIN_TOKENS = ["D"]

# Key waypoint tokens (intermediate dissolution pathway)
WAYPOINT_TOKENS = ["local", "national", "RAW", "BR", "AB",
                    "def", "GR", "MA", "MC", "sh"]

ALL_TOKENS = BASIN_TOKENS + WAYPOINT_TOKENS

# Extract token IDs
tokenizer = model.tokenizer

md = "## Token ID Registry\n\n"
md += "| Token | BPE Index | Type | Full Decode Check |\n"
md += "|:---|:---|:---|:---|\n"

token_ids = {}
for token_str in ALL_TOKENS:
    ids_no_space = tokenizer.encode(token_str, add_special_tokens=False)
    ids_with_space = tokenizer.encode(" " + token_str, add_special_tokens=False)
    
    if len(ids_no_space) == 1:
        tid = ids_no_space[0]
    elif len(ids_with_space) == 1:
        tid = ids_with_space[0]
    else:
        tid = ids_no_space[0]
    
    decode_check = tokenizer.decode([tid])
    token_type = "BASIN" if token_str in BASIN_TOKENS else "WAYPOINT"
    token_ids[token_str] = tid
    
    md += f"| `{token_str}` | {tid} | {token_type} | `{repr(decode_check)}` |\n"

md += f"\n*Vocabulary size: {model.cfg.d_vocab} tokens*\n"
display(Markdown(md))

print("\nToken IDs dict:")
for k, v in token_ids.items():
    print(f"  {k:>12s} → {v}")

# === SAVE ===
with open(OUTPUT_DIR / "data" / "token_ids.json", "w") as f:
    json.dump(token_ids, f, indent=2)
print(f"\n✅ Saved token IDs to {OUTPUT_DIR / 'data' / 'token_ids.json'}")

---
## 3. Embedding Vectors

Extract the raw embedding vectors from `W_E` and compute basic properties.

In [ ]:
# ============================================================
# STEP 3: EMBEDDING PROPERTIES
# ============================================================

W_E = model.W_E.detach().cpu()  # [vocab_size, d_model]

all_norms = W_E.norm(dim=1)
mean_norm = all_norms.mean().item()
std_norm = all_norms.std().item()

norm_ranks = all_norms.argsort(descending=True)
rank_lookup = {tid.item(): rank for rank, tid in enumerate(norm_ranks)}

md = "## Embedding Properties\n\n"
md += f"**Vocabulary stats:** mean norm = {mean_norm:.4f}, std = {std_norm:.4f}\n\n"
md += "| Token | BPE ID | Embedding Norm | Z-score | Norm Rank | Outlier? |\n"
md += "|:---|:---|:---|:---|:---|:---|\n"

embed_props = {}
for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    norm = all_norms[tid].item()
    z_score = (norm - mean_norm) / std_norm
    rank = rank_lookup[tid] + 1
    percentile = (rank / model.cfg.d_vocab) * 100
    outlier = "⚠ YES" if abs(z_score) > 2.0 else ""
    md += f"| `{token_str}` | {tid} | {norm:.4f} | {z_score:+.2f} | {rank} ({percentile:.1f}%) | {outlier} |\n"
    embed_props[token_str] = {
        "id": tid, "norm": round(norm, 4), "z_score": round(z_score, 2),
        "rank": rank, "percentile": round(percentile, 1)
    }

display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "embedding_properties.json", "w") as f:
    json.dump({"mean_norm": mean_norm, "std_norm": std_norm, "tokens": embed_props}, f, indent=2)
print(f"✅ Saved embedding properties to {OUTPUT_DIR / 'data' / 'embedding_properties.json'}")

---
## 4. Embedding Neighbourhood Analysis

For each basin/waypoint token, find the **20 nearest neighbours** in the full `W_E` space.

### What we're looking for:
- **Semantic neighbours** (e.g., `D` → `C`, `E`, `F` if alphabetic) → semantic clustering
- **BPE substring neighbours** (e.g., `D` → `Da`, `De`) → substring adjacency
- **Random/anomalous neighbours** → suggests glitch token properties

In [ ]:
# ============================================================
# STEP 4: NEAREST NEIGHBOURS IN W_E
# ============================================================

def get_nearest_neighbours(model, token_id, W_E, k=20):
    """Find k nearest neighbours by cosine similarity in W_E."""
    target_vec = W_E[token_id].unsqueeze(0)
    norms = W_E.norm(dim=1, keepdim=True).clamp(min=1e-8)
    W_E_normed = W_E / norms
    target_normed = target_vec / target_vec.norm().clamp(min=1e-8)
    sims = (W_E_normed @ target_normed.T).squeeze()
    top_sims, top_ids = torch.topk(sims, k + 1)
    results = []
    for sim, idx in zip(top_sims[1:], top_ids[1:]):
        decoded = model.tokenizer.decode([idx.item()])
        results.append({"id": idx.item(), "token": decoded, "cosine_sim": round(sim.item(), 4)})
    return results


all_neighbours = {}
for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    token_type = "BASIN" if token_str in BASIN_TOKENS else "WAYPOINT"
    neighbours = get_nearest_neighbours(model, tid, W_E, k=20)
    all_neighbours[token_str] = neighbours
    
    md = f"### `{token_str}` (ID: {tid}, {token_type})\n\n"
    md += "| Rank | Neighbour | ID | Cosine Sim | BPE Prefix? |\n"
    md += "|:---|:---|:---|:---|:---|\n"
    
    for i, n in enumerate(neighbours):
        clean_tok = n['token'].replace('\n', '↵').replace('|', '∣')
        prefix_match = "✓" if (len(token_str) >= 3 and 
                               len(clean_tok.strip()) >= 3 and
                               clean_tok.strip()[:3].lower() == token_str[:3].lower()) else ""
        md += f"| {i+1} | `{clean_tok}` | {n['id']} | {n['cosine_sim']:.4f} | {prefix_match} |\n"
    
    md += "\n"
    display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "neighbours.json", "w", encoding="utf-8") as f:
    json.dump(all_neighbours, f, indent=2, ensure_ascii=False)
print(f"✅ Saved all neighbour data to {OUTPUT_DIR / 'data' / 'neighbours.json'}")

---
## 5. Cross-Similarity: Basin & Waypoint Token Matrix

How similar are the basin/waypoint tokens to *each other* in embedding space?

In [ ]:
# ============================================================
# STEP 5: BASIN/WAYPOINT CROSS-SIMILARITY
# ============================================================
import plotly.express as px
import plotly.io as pio

ids_list = [token_ids[t] for t in ALL_TOKENS]
embeddings = W_E[ids_list]
norms = embeddings.norm(dim=1, keepdim=True).clamp(min=1e-8)
embeddings_normed = embeddings / norms
sim_matrix = (embeddings_normed @ embeddings_normed.T).numpy()

labels = [f"{'B' if t in BASIN_TOKENS else 'W'}:{t}" for t in ALL_TOKENS]

fig = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="GPT-2 Medium: Basin & Waypoint Token Similarity in W_E",
    text_auto=".2f",
    aspect="auto",
)
fig.update_layout(template="plotly_dark", height=700, width=700)
fig.show()

# === SAVE ===
pio.write_image(fig, str(OUTPUT_DIR / "images" / "cross_similarity_matrix.png"), scale=2)
fig.write_html(str(OUTPUT_DIR / "images" / "cross_similarity_matrix.html"))
np.save(str(OUTPUT_DIR / "data" / "cross_similarity_matrix.npy"), sim_matrix)
with open(OUTPUT_DIR / "data" / "cross_similarity_labels.json", "w") as f:
    json.dump(labels, f)
print(f"✅ Saved cross-similarity matrix (PNG, HTML, NPY) to {OUTPUT_DIR}")

---
## 6. Known Glitch Token Check

In [ ]:
# ============================================================
# STEP 6: GLITCH TOKEN DIAGNOSTICS
# ============================================================

KNOWN_GLITCH_TOKENS = [
    " SolidGoldMagworthy", " petertodd", " StreamerBot",
    " TheNitromeFan", " davidjl", " guaneletters",
    " RandomReddworthy", " embedreportprint",
    " rawdownloadcloneembedreportprint",
    " SolidGoldMag", " exaboraliverably",
]

glitch_ids = set()
for gt in KNOWN_GLITCH_TOKENS:
    ids = tokenizer.encode(gt, add_special_tokens=False)
    glitch_ids.update(ids)

glitch_results = {}
md = "## Glitch Token Check\n\n"
for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    is_glitch = tid in glitch_ids
    status = "⚠ YES" if is_glitch else "✓ No"
    md += f"- `{token_str}` (ID {tid}): {status}\n"
    glitch_results[token_str] = {"id": tid, "is_glitch": is_glitch}

md += "\n### Additional Diagnostics\n\n"
md += "| Token | ID | Norm | vs Mean | Norm Rank |\n"
md += "|:---|:---|:---|:---|:---|\n"

for token_str in ALL_TOKENS:
    tid = token_ids[token_str]
    norm = all_norms[tid].item()
    rank = rank_lookup[tid] + 1
    percentile = (rank / model.cfg.d_vocab) * 100
    md += f"| `{token_str}` | {tid} | {norm:.4f} | {norm/mean_norm:.2f}x | {rank} ({percentile:.1f}%) |\n"

display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "glitch_check.json", "w") as f:
    json.dump(glitch_results, f, indent=2)
print(f"✅ Saved glitch check to {OUTPUT_DIR / 'data' / 'glitch_check.json'}")

---
## 7. Summary

In [ ]:
# ============================================================
# STEP 7: SUMMARY
# ============================================================

# Count BPE prefix matches across all basin token neighbours
bpe_counts = {}
semantic_counts = {}
for token_str in BASIN_TOKENS:
    neighbours = all_neighbours[token_str]
    bpe_match = sum(1 for n in neighbours 
                    if len(token_str) >= 2 and len(n['token'].strip()) >= 2 
                    and n['token'].strip()[:2].lower() == token_str[:2].lower())
    bpe_counts[token_str] = bpe_match
    semantic_counts[token_str] = 20 - bpe_match

summary = {
    "model": "gpt2-medium",
    "analysis": "Token neighbourhood in W_E",
    "date": "2026-03-25",
    "basin_tokens": BASIN_TOKENS,
    "waypoint_tokens": WAYPOINT_TOKENS,
    "bpe_prefix_matches": bpe_counts,
    "non_prefix_neighbours": semantic_counts,
    "total_tokens_analysed": len(ALL_TOKENS),
}

md = "## Summary: GPT-2 Medium Token Neighbourhood Analysis\n\n"
md += "| Basin Token | BPE Prefix Matches (of 20) | Non-Prefix Neighbours |\n"
md += "|:---|:---|:---|\n"
for t in BASIN_TOKENS:
    md += f"| `{t}` | {bpe_counts[t]} | {semantic_counts[t]} |\n"

md += "\n### Observations\n\n"
md += "*To be filled after examining the neighbour tables above.*\n"

display(Markdown(md))

# === SAVE ===
with open(OUTPUT_DIR / "data" / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

with open(OUTPUT_DIR / "SUMMARY.md", "w", encoding="utf-8") as f:
    f.write("# Token Neighbourhood Analysis: GPT-2 Medium\n\n")
    f.write(f"**Date:** 2026-03-25\n")
    f.write(f"**Tokens analysed:** {len(ALL_TOKENS)} ({len(BASIN_TOKENS)} basin, {len(WAYPOINT_TOKENS)} waypoint)\n\n")
    f.write("## Saved Files\n\n")
    f.write("| File | Contents |\n|:---|:---|\n")
    f.write("| `data/token_ids.json` | BPE indices for all tokens |\n")
    f.write("| `data/embedding_properties.json` | Norms, z-scores, ranks |\n")
    f.write("| `data/neighbours.json` | 20 nearest neighbours per token |\n")
    f.write("| `data/cross_similarity_matrix.npy` | Cross-similarity matrix |\n")
    f.write("| `data/glitch_check.json` | Glitch token check results |\n")
    f.write("| `data/summary.json` | Summary with BPE prefix counts |\n")
    f.write("| `images/cross_similarity_matrix.png` | Heatmap (2x resolution) |\n")
    f.write("| `images/cross_similarity_matrix.html` | Interactive heatmap |\n")

print(f"✅ Saved summary to {OUTPUT_DIR / 'SUMMARY.md'}")
print(f"\n🏁 Token neighbourhood analysis complete. All outputs in: {OUTPUT_DIR.resolve()}")